<h1 style="text-align: center; font-size: 48px; color: #008B9B; font-weight: bold;">Film Search App — DB "sakila"</h1>

In [1]:
# Imports
import pymysql
from pymongo import MongoClient
from datetime import datetime, timezone
from tabulate import tabulate
from IPython.display import clear_output, HTML, display
from dotenv import load_dotenv
import os

# Load environment variables from .env file
load_dotenv()

# Global constants
SEP = "*" * 79
PAGE_SIZE = 10  # number of rows per pagination page


# Container class for terminal color and style codes
class Style:
    # Effects
    RESET = '\033[0m'
    BOLD = '\033[1m'
    DIM = '\033[2m'
    UNDERLINE = '\033[4m'

    # Text colors
    RED = '\033[31m'
    GREEN = '\033[32m'
    YELLOW = '\033[33m'
    BLUE = '\033[34m'
    CYAN = '\033[36m'
    MAGENTA = '\033[35m'


# ---------------------------------------------------------------------------
# 1. MySQL — connection config and connection test
# ---------------------------------------------------------------------------

config = {
    "host": os.getenv("MYSQL_HOST"),
    "user": os.getenv("MYSQL_USER"),
    "password": os.getenv("MYSQL_PASSWORD"),
    "database": os.getenv("MYSQL_DB"),
    "charset": "utf8mb4",
    "cursorclass": pymysql.cursors.DictCursor,
    "connect_timeout": 5  # seconds to wait before timing out
}


def test_mysql_connection():
    """Checks the MySQL connection and returns a colored status string."""
    try:
        with pymysql.connect(**config) as conn:
            with conn.cursor() as cur:
                cur.execute("SELECT 1")
        return f"{Style.GREEN}● MySQL: CONNECTED{Style.RESET}"
    except Exception as e:
        # Error message is dimmed to visually separate it from the status label
        return (
            f"{Style.RED}{Style.BOLD}● MySQL: FAILED {Style.RESET}"
            f"{Style.RED}{Style.DIM}({e}){Style.RESET}"
        )


# ---------------------------------------------------------------------------
# 2. Search functions (MySQL) with pagination
# ---------------------------------------------------------------------------

# --- 2.1 Helper functions ---

def get_all_genres():
    """Fetches a list of all movie genres sorted alphabetically.

    Returns:
        list: A list of genre name strings.
    """
    with pymysql.connect(**config) as conn:
        with conn.cursor() as cur:
            # Returns only genre names, sorted A–Z
            cur.execute("SELECT name FROM category ORDER BY name")
            rows = cur.fetchall()

    # Convert list of dicts [{"name": "Action"}, ...] to plain list ["Action", ...]
    return [row["name"] for row in rows]


def get_min_max_years():
    """Fetches the minimum and maximum release years from the film table.

    Returns:
        tuple: (min_year, max_year)
    """
    query = "SELECT MIN(release_year) AS year_min, MAX(release_year) AS year_max FROM film;"

    with pymysql.connect(**config) as conn:
        with conn.cursor() as cur:
            cur.execute(query)
            row = cur.fetchone()

    return row["year_min"], row["year_max"]


def get_year_range_input():
    """Collects and validates a movie release year range (From and To) from the user.

    Uses the actual year boundaries from the Sakila database.
    Returns:
        dict: May contain 'year_from', 'year_to', or both, or be empty.
    """
    year_params = {}

    # Fetch current year boundaries directly from MySQL
    min_db_year, max_db_year = get_min_max_years()
    print("\nSelect year range:")

    def _ask_year(label):
        """Prompts and validates a single year value (From or To).

        Returns:
            int if the year is valid, None if the user pressed Enter to skip.
        """
        while True:
            year = input(
                f"Year {label} (database range {min_db_year}-{max_db_year}, "
                f"or Enter to skip): "
            ).strip()

            if not year:
                return None

            if year.isdigit() and len(year) == 4:
                year_int = int(year)
                if min_db_year <= year_int <= max_db_year:
                    return year_int
                print(
                    f"{Style.BOLD}{Style.RED}Error: the Sakila database only contains films "
                    f"from {min_db_year} to {max_db_year}!{Style.RESET}"
                )
                continue

            print(
                f"{Style.BOLD}{Style.RED}Error: please enter a valid "
                f"four-digit year (e.g. 2006)!{Style.RESET}"
            )

    # --- Year FROM ---
    year_from = _ask_year("from")
    if year_from is not None:
        year_params["year_from"] = year_from

    # --- Year TO ---
    while True:
        year_to = _ask_year("to")
        if year_to is None:
            break
        # Guard: "to" year cannot be earlier than "from" year
        if "year_from" in year_params and year_to < year_params["year_from"]:
            print(
                f"{Style.BOLD}{Style.RED}Error: 'to' year cannot be "
                f"less than 'from' year!{Style.RESET}"
            )
            continue
        year_params["year_to"] = year_to
        break

    return year_params


def get_actors_by_keyword(actor_input):
    """Searches for actors in the database by a fragment of their first or last name.

    Args:
        actor_input (str): Keyword entered by the user.
    Returns:
        list: A list of dicts with actor_id and actor_name.
    """
    with pymysql.connect(**config) as conn:
        with conn.cursor() as cur:
            # Matches both first and last name; MySQL LIKE is case-insensitive
            query = """
                SELECT actor_id, CONCAT(first_name, ' ', last_name) AS actor_name
                FROM actor
                WHERE first_name LIKE %s OR last_name LIKE %s
                ORDER BY first_name, last_name;
            """
            search_pattern = f"%{actor_input}%"
            cur.execute(query, (search_pattern, search_pattern))
            rows = cur.fetchall()

    return rows


def get_search_inputs(search_type):
    """Collects and validates user input for the given search type.

    Displays numbered lists where needed and guards against invalid input.

    Args:
        search_type (str): One of "keyword", "genres-years", "actor".
    Returns:
        dict: Validated search parameters, or empty dict if user cancelled.
    """

    # ------------------------------------------------------------------
    # Option 1: Search by film title keyword
    # ------------------------------------------------------------------
    if search_type == "keyword":
        keyword = input("Enter part of a film title (or Enter to skip): ").strip().lower()
        return {"keyword": keyword} if keyword else {}

    # ------------------------------------------------------------------
    # Option 2: Search by genre and year range
    # ------------------------------------------------------------------
    if search_type == "genres-years":
        params = {}

        # Show the full genre list before input (required by the spec)
        all_genres = get_all_genres()
        print(f"\n{SEP}")
        print(f"{Style.BOLD}{Style.MAGENTA}=== ALL AVAILABLE GENRES ==={Style.RESET}")
        print(SEP)

        for idx, g in enumerate(all_genres, 1):  # idx — index, g — genre name
            print(f"{idx:2d}. {g}")  # :2d aligns single and double-digit numbers
        print(SEP)

        # Genre selection by number
        while True:
            choice = input("\nSelect genre number (or Enter to skip): ").strip()

            if not choice:  # Enter — skip genre filter
                break

            if choice.isdigit() and 1 <= int(choice) <= len(all_genres):
                params["genre"] = all_genres[int(choice) - 1]
                break
            else:
                print(
                    f"{Style.BOLD}{Style.RED}Error: enter a number from 1 to "
                    f"{len(all_genres)}!{Style.RESET}"
                )

        # Collect and validate year range using the shared helper
        year_data = get_year_range_input()

        if "year_from" in year_data:
            params["year_from"] = year_data["year_from"]
        if "year_to" in year_data:
            params["year_to"] = year_data["year_to"]

        return params

    # ------------------------------------------------------------------
    # Option 3: Search by actor and year range
    # ------------------------------------------------------------------
    if search_type == "actor":
        params = {}

        while True:
            actor_input = input(
                "Enter actor first name, last name, or part of a name (or Enter to cancel): "
            ).strip()

            if not actor_input:
                # User pressed Enter — cancel search and return empty dict
                return {}

            found_actors = get_actors_by_keyword(actor_input)

            if not found_actors:
                print("Actor not found. Try again (e.g. 'Nick' or 'Wahlberg').")
                continue

            # Show paginated actor list and let user select one
            selected = paginate_actors(found_actors)

            if selected is None:
                # User pressed Enter to search again
                continue

            params["actor_id"] = selected["actor_id"]
            params["actor_name"] = selected["actor_name"]
            break

        # Year range filter also applies to actor search for flexible time-based browsing
        year_data = get_year_range_input()

        if "year_from" in year_data:
            params["year_from"] = year_data["year_from"]
        if "year_to" in year_data:
            params["year_to"] = year_data["year_to"]

        return params


def build_search_summary(search_type, params, total_found):
    """Builds a summary header string with the search description and total result count.

    Shown above the results table when there is more than one page.

    Args:
        search_type (str): One of "keyword", "genres-years", "actor".
        params (dict): Search parameters.
        total_found (int): Total number of results found.
    Returns:
        str: Ready-to-print summary string.
    """
    # Build the time period description
    if "year_from" in params and "year_to" in params:
        period = f"for the period {params['year_from']}–{params['year_to']}"
    elif "year_from" in params:
        period = f"from {params['year_from']}"
    elif "year_to" in params:
        period = f"up to {params['year_to']}"
    else:
        period = ""

    # Build the main description based on search type
    if search_type == "keyword":
        keyword = params.get("keyword", "")
        base = f"Total films matching '{keyword}'"
    elif search_type == "genres-years":
        genre = params.get("genre", "all genres")
        base = f"Total films in genre '{genre}'"
    elif search_type == "actor":
        actor = params.get("actor_name", "selected actor")
        base = f"Total films for '{actor}'"
    else:
        base = "Total results"

    if period:
        return f"{base} {period}: {total_found}"
    return f"{base}: {total_found}"


def paginate_results(results, summary=None):
    """Displays query results in pages of PAGE_SIZE rows.

    Allows selecting ANY film that has already been shown on screen.
    Clears the Jupyter output only when returning to the main menu.

    Args:
        results (list): Full list of film dicts from the database.
        summary (str): Optional summary header shown when results span multiple pages.
    Returns:
        int: Total number of films the user viewed.
    """
    total_found = len(results)
    if total_found == 0:
        print("\nNo results found.")
        return 0

    if summary and total_found > PAGE_SIZE:
        print(f"\n{SEP}\n  {summary}\n{SEP}")
    else:
        print(f"\nResults found: {total_found}")

    total_viewed = 0
    current_index = 0
    show_table = True

    while current_index < total_found:
        chunk = results[current_index: current_index + PAGE_SIZE]
        current_start = current_index + 1

        if show_table:
            display_results(chunk, start_index=current_start)

        print(f"\n{SEP}")
        print(f"{Style.BOLD}{Style.MAGENTA}AVAILABLE ACTIONS (FILMS):{Style.RESET}")
        print("  [Film number]  - View detailed film description")
        if current_index + PAGE_SIZE < total_found:
            print("  y              - Show next 10 results")
        print("  Enter          - Return to main menu (leave empty)")
        print(SEP)

        action_taken = None

        while True:
            # Count how many films the user has seen so far
            max_displayed = current_index + len(chunk)

            choice = input(
                f"Enter command (showing {max_displayed} of {total_found}): "
            ).strip().lower()

            if not choice:  # Enter — clear screen and exit to menu
                total_viewed += len(chunk)
                clear_output(wait=False)
                return total_viewed

            if choice.isdigit():
                film_number = int(choice)

                # Accept any number from 1 to max_displayed (not just the current page)
                if 1 <= film_number <= max_displayed:
                    # Find which page this film belongs to
                    target_global_idx = film_number - 1
                    orig_chunk_start = (target_global_idx // PAGE_SIZE) * PAGE_SIZE
                    orig_chunk = results[orig_chunk_start: orig_chunk_start + PAGE_SIZE]
                    orig_start_index = orig_chunk_start + 1

                    select_film_from_list(orig_chunk, choice, orig_start_index)
                    action_taken = "stay"
                    break
                else:
                    print(
                        f"{Style.BOLD}{Style.RED}Error: you can select any displayed film "
                        f"from 1 to {max_displayed}!{Style.RESET}"
                    )
                    continue

            if choice == "y" and current_index + PAGE_SIZE < total_found:
                action_taken = "next"
                break

            print(
                f"{Style.BOLD}{Style.RED}Invalid command. Enter a valid film number, "
                f"'y', or press Enter to exit.{Style.RESET}"
            )

        if action_taken == "next":
            total_viewed += len(chunk)
            current_index += PAGE_SIZE
            show_table = True
        elif action_taken == "stay":
            show_table = False

    clear_output(wait=False)
    return total_viewed


def paginate_actors(actors):
    """Paginates the list of found actors in chunks of PAGE_SIZE.

    Allows selecting ANY actor already shown on screen.

    Args:
        actors (list): List of actor dicts with actor_id and actor_name.
    Returns:
        dict: The selected actor dict, or None if the user cancelled.
    """
    total_actors = len(actors)
    current_index = 0

    while current_index < total_actors:
        chunk = actors[current_index: current_index + PAGE_SIZE]
        current_start = current_index + 1

        print(f"\n{SEP}")
        print(f"{Style.BOLD}{Style.MAGENTA}ACTORS FOUND:{Style.RESET}")
        print(SEP)
        for idx, actor in enumerate(chunk, start=current_start):
            print(f"  {idx:2d}. {actor['actor_name']}")
        print(SEP)

        print(f"{Style.BOLD}{Style.MAGENTA}AVAILABLE ACTIONS (ACTORS):{Style.RESET}")
        print("  [Actor number] - Select actor to search their films")
        if current_index + PAGE_SIZE < total_actors:
            print("  y              - Show next 10 actors")
        print("  Enter          - Return to name input (search again)")
        print(SEP)

        action_taken = None

        while True:
            # Count how many actors the user has seen so far
            max_displayed = current_index + len(chunk)

            choice = input(
                f"Enter command (showing {max_displayed} of {total_actors}): "
            ).strip().lower()

            if not choice:  # Enter — clear screen and go back to name search
                clear_output(wait=False)
                return None

            if choice.isdigit():
                actor_num = int(choice)

                # Accept any number from 1 to max_displayed
                if 1 <= actor_num <= max_displayed:
                    clear_output(wait=False)
                    return actors[actor_num - 1]
                else:
                    print(
                        f"{Style.BOLD}{Style.RED}Error: you can select any displayed actor "
                        f"from 1 to {max_displayed}!{Style.RESET}"
                    )
                    continue

            if choice == "y" and current_index + PAGE_SIZE < total_actors:
                action_taken = "next"
                break

            print(
                f"{Style.BOLD}{Style.RED}Invalid command. Enter a valid actor number, "
                f"'y', or press Enter.{Style.RESET}"
            )

        if action_taken == "next":
            current_index += PAGE_SIZE

    # Reached the end of the list without a selection
    clear_output(wait=False)
    return None


def build_sql_filters(params):
    """Dynamically builds a WHERE clause and argument list for MySQL queries.

    Args:
        params (dict): Dictionary with search filters.
    Returns:
        tuple: (where_clause_str, query_args_list)
    """
    if not params:
        return "", []

    conditions = []
    query_args = []

    # 1. Filter by keyword in film title
    if "keyword" in params:
        conditions.append("f.title LIKE %s")
        query_args.append(f"%{params['keyword']}%")

    # 2. Filter by genre name
    if "genre" in params:
        conditions.append("c.name = %s")
        query_args.append(params["genre"])

    # 3. Filter by actor ID (exact ID selected by the user)
    if "actor_id" in params:
        conditions.append("fa.actor_id = %s")
        query_args.append(params["actor_id"])

    # 4. Year range filters — .extend() unpacks [from, to] into two separate values
    #    so the SQL query receives two clean integers, not an array
    if "year_from" in params and "year_to" in params:
        conditions.append("f.release_year BETWEEN %s AND %s")
        query_args.extend([params["year_from"], params["year_to"]])
    elif "year_from" in params:
        conditions.append("f.release_year >= %s")
        query_args.append(params["year_from"])
    elif "year_to" in params:
        conditions.append("f.release_year <= %s")
        query_args.append(params["year_to"])

    # 5. Filter by exact single year (optional, for actor search)
    if "year" in params:
        conditions.append("f.release_year = %s")
        query_args.append(params["year"])

    # Combine all conditions with AND
    where_clause = " WHERE " + " AND ".join(conditions)

    return where_clause, query_args


def get_sort_direction():
    """Asks the user for sort direction by release year.

    Returns:
        str: 'DESC' (newest first) or 'ASC' (oldest first).
    """
    while True:
        print(f"\n{Style.BOLD}{Style.MAGENTA}--- SORT ORDER ---{Style.RESET}")
        print("1. Newest first (descending)")
        print("2. Oldest first (ascending)")

        choice = input("Your choice (default 1): ").strip()

        if choice == "2":
            return "ASC"
        if choice == "1" or not choice:
            return "DESC"

        print(f"{Style.BOLD}{Style.RED}Error: enter 1 or 2!{Style.RESET}")


def select_film_from_list(current_films, user_choice, start_index):
    """Finds a film in the current page chunk by user-entered number and displays its description.

    Args:
        current_films (list): The page slice containing the selected film.
        user_choice (str): The number the user entered.
        start_index (int): The global starting number of this chunk (e.g. 1, 11, 21).
    """
    try:
        # Calculate position within the chunk (e.g. film 17 in chunk starting at 11: 17 - 11 = 6)
        idx = int(user_choice) - start_index

        if 0 <= idx < len(current_films):
            selected_film = current_films[idx]
            film_id = selected_film.get("film_id")
            title = selected_film.get("title", "Unknown")

            description = get_film_description(film_id)

            # Separator matching SEP length, using "=" for visual contrast
            BOX_SEP = "=" * len(SEP)

            print(f"\n{Style.BOLD}{Style.YELLOW}{BOX_SEP}{Style.RESET}")
            print(f"{Style.BOLD}{Style.YELLOW}🎬 FILM DESCRIPTION: {title.upper()}{Style.RESET}")
            print(f"{Style.BOLD}{Style.YELLOW}{BOX_SEP}{Style.RESET}")
            print(description)
            print(f"{Style.BOLD}{Style.YELLOW}{BOX_SEP}{Style.RESET}\n")
        else:
            print(
                f"{Style.BOLD}{Style.RED}Error: number must be between "
                f"{start_index} and {start_index + len(current_films) - 1}.{Style.RESET}"
            )

    except ValueError:
        print(
            f"{Style.BOLD}{Style.RED}Error: please enter "
            f"a valid numeric film number.{Style.RESET}"
        )


# --- 2.2 Core search functions ---

def search_film_by_keyword():
    """Searches for films by title keyword using the universal SQL filter builder."""

    # 1. Get user input
    params = get_search_inputs("keyword")
    if not params:
        print("Search query is empty. Returning to menu.")
        return

    # 2. Build SQL WHERE clause
    where_clause, query_args = build_sql_filters(params)

    base_query = """
        SELECT f.film_id, f.title, c.name AS genre, f.release_year, f.rating
        FROM film f
        JOIN film_category fc ON f.film_id = fc.film_id
        JOIN category c ON fc.category_id = c.category_id
    """
    final_query = base_query + where_clause + ";"

    # 3. Execute query
    try:
        with pymysql.connect(**config) as conn:
            with conn.cursor() as cur:
                cur.execute(final_query, query_args)
                films = cur.fetchall()
    except Exception as e:
        print(f"{Style.BOLD}{Style.RED}MySQL error: {e}{Style.RESET}")
        return

    # Guard against empty results
    if not films:
        print("\n❌ No films found in the Sakila database for your query.")
        # Log the empty query — the system records all searches, including empty ones
        log_query("keyword", params, 0, 0)
        return

    # 4. Pass results to shared pagination
    summary = build_search_summary("keyword", params, len(films))
    total_viewed = paginate_results(films, summary=summary)

    # 5. Log to MongoDB
    log_query("keyword", params, len(films), total_viewed)
    print(f"\n[MongoDB] Log saved. Found: {len(films)}, Viewed: {total_viewed}")


def search_film_by_genre_and_years():
    """Searches for films by selected genre and year range (sorted by year descending)."""

    # 1. Get user-selected parameters
    params = get_search_inputs("genres-years")
    if not params:
        print("Nothing selected. Returning to menu.")
        return

    # 2. Build SQL WHERE clause
    where_clause, query_args = build_sql_filters(params)

    base_query = """
        SELECT f.film_id, f.title, c.name AS genre, f.release_year, f.rating
        FROM film f
        JOIN film_category fc ON f.film_id = fc.film_id
        JOIN category c ON fc.category_id = c.category_id
    """
    final_query = base_query + where_clause + " ORDER BY f.release_year DESC;"

    # 3. Execute query
    try:
        with pymysql.connect(**config) as conn:
            with conn.cursor() as cur:
                cur.execute(final_query, query_args)
                films = cur.fetchall()
    except Exception as e:
        print(f"{Style.BOLD}{Style.RED}MySQL error: {e}{Style.RESET}")
        return

    # Guard against empty results
    if not films:
        print("\n❌ No films found in the Sakila database for your query.")
        # Log even empty queries — the system records all searches
        log_query("genres-years", params, 0, 0)
        return

    # 4. Paginate and log
    summary = build_search_summary("genres-years", params, len(films))
    total_viewed = paginate_results(films, summary=summary)
    log_query("genres-years", params, len(films), total_viewed)
    print(f"\n[MongoDB] Log saved. Found: {len(films)}, Viewed: {total_viewed}")


def search_film_by_actor_and_year():
    """Searches for films by selected actor and year range."""

    # 1. Get actor parameters from user input
    params = get_search_inputs("actor")
    if not params:
        print("Nothing selected. Returning to menu.")
        return

    # 2. Build SQL WHERE clause and ask for sort order
    where_clause, query_args = build_sql_filters(params)
    sort_dir = get_sort_direction()

    base_query = """
        SELECT f.film_id, f.title, c.name AS genre, f.release_year, f.rating
        FROM film f
        JOIN film_actor fa ON f.film_id = fa.film_id
        JOIN film_category fc ON f.film_id = fc.film_id
        JOIN category c ON fc.category_id = c.category_id
    """
    # Dynamically insert ASC or DESC
    final_query = base_query + where_clause + f" ORDER BY f.release_year {sort_dir};"

    # 3. Execute query
    try:
        with pymysql.connect(**config) as conn:
            with conn.cursor() as cur:
                cur.execute(final_query, query_args)
                films = cur.fetchall()
    except Exception as e:
        print(f"{Style.BOLD}{Style.RED}MySQL error: {e}{Style.RESET}")
        return

    # Guard against empty results
    if not films:
        print("\n❌ No films found in the Sakila database for your query.")
        log_query("actor-years", params, 0, 0)
        return

    print(f"\nFilmography: {params.get('actor_name', 'Selected actor')}")

    # 4. Paginate and log
    summary = build_search_summary("actor", params, len(films))
    total_viewed = paginate_results(films, summary=summary)
    log_query("actor-years", params, len(films), total_viewed)
    print(f"\n[MongoDB] Log saved. Found: {len(films)}, Viewed: {total_viewed}")


def get_film_description(film_id):
    """Returns the text description of a film from MySQL by its ID.

    Args:
        film_id (int): Film ID from the film table.
    Returns:
        str: Film description text, or a message if not available.
    """
    query = """SELECT description
               FROM film
               WHERE film_id = %s"""
    try:
        with pymysql.connect(**config) as conn:
            with conn.cursor() as cur:
                cur.execute(query, (film_id,))
                result = cur.fetchone()
                if result and result.get("description"):
                    return result["description"]
                return "No description available for this film."
    except Exception as e:
        print(
            f"{Style.BOLD}{Style.RED}[MySQL Error] Could not fetch film description: "
            f"{e}{Style.RESET}"
        )
        return "Could not load description due to a technical error."


def display_results(films, start_index=1):
    """Displays a list of films as a formatted table using tabulate.

    Args:
        films (list): List of film dicts from the database.
        start_index (int): The number to start numbering rows from.
    """
    if not films:
        print("No films found.")
        return

    # Bold cyan column headers
    headers = [
        f"{Style.BOLD}{Style.CYAN}#{Style.RESET}",
        f"{Style.BOLD}{Style.CYAN}Title{Style.RESET}",
        f"{Style.BOLD}{Style.CYAN}Genre{Style.RESET}",
        f"{Style.BOLD}{Style.CYAN}Year{Style.RESET}",
        f"{Style.BOLD}{Style.CYAN}Rating{Style.RESET}"
    ]

    # Build plain text rows (no color codes — keeps table alignment clean)
    table_data = []
    for index, film in enumerate(films, start=start_index):
        table_data.append([
            index,
            film['title'],
            film['genre'],
            film['release_year'],
            film['rating']
        ])

    # fancy_grid gives a clean bordered table style
    print(tabulate(
        table_data, headers=headers,
        tablefmt="fancy_grid", stralign="left", numalign="right"
    ))


# ---------------------------------------------------------------------------
# 3. MongoDB — connection config and connection test
# ---------------------------------------------------------------------------

# Initialize the MongoDB client ONCE at module level;
# pymongo manages the connection pool automatically
MONGO_CONFIG = {
    "host": os.getenv("MONGO_HOST"),
    "username": os.getenv("MONGO_USER"),
    "password": os.getenv("MONGO_PASSWORD"),
    "authSource": os.getenv("MONGO_AUTH"),
}
MONGO_DB_NAME = os.getenv("MONGO_DB")
COLLECTION_NAME = os.getenv("COLLECTION")

mongo_client = MongoClient(**MONGO_CONFIG)
db_collection = mongo_client[MONGO_DB_NAME][COLLECTION_NAME]

# Single source of truth for search type display labels (used in stats tables)
SEARCH_TYPE_LABELS = {
    "keyword": "By keyword",
    "genres-years": "By genre / years",
    "actor": "By actor",
    "actor-years": "By actor / years"
}


def test_mongo_connection():
    """Checks the MongoDB connection and returns a colored status string."""
    try:
        mongo_client.server_info()
        return f"{Style.GREEN}● MongoDB: CONNECTED{Style.RESET}"
    except Exception as e:
        return (
            f"{Style.RED}{Style.BOLD}● MongoDB: FAILED {Style.RESET}"
            f"{Style.RED}{Style.DIM}({e}){Style.RESET}"
        )


# ---------------------------------------------------------------------------
# 4. MongoDB — query logging
# ---------------------------------------------------------------------------

def log_query(search_type, params, total_found, total_viewed):
    """Records search query parameters to the MongoDB collection.

    Args:
        search_type (str): Type of search performed.
        params (dict): Search parameters used.
        total_found (int): Total number of results returned by the query.
        total_viewed (int): Number of results the user actually viewed.
    """
    try:
        document = {
            # Stored as a Python datetime — MongoDB converts it to BSON Date automatically
            "timestamp": datetime.now(timezone.utc),
            "search_type": search_type,
            "params": params,
            "results_count": total_found,
            "total_viewed": total_viewed
        }
        db_collection.insert_one(document)
    except Exception as e:
        print(f"\n[MongoDB Error] Could not save log: {e}")


def get_top_queries():
    """Returns the 5 most frequently repeated search queries using an aggregation pipeline."""
    try:
        pipeline = [
            {"$group": {
                "_id": {"type": "$search_type", "params": "$params"},
                "count": {"$sum": 1},
                "avg_found": {"$avg": "$results_count"},
                "avg_viewed": {"$avg": "$total_viewed"}
            }},
            {"$sort": {"count": -1}},
            {"$limit": 5}
        ]
        return list(db_collection.aggregate(pipeline))
    except Exception as e:
        print(f"\n[MongoDB Error] Could not fetch top queries: {e}")
        return []


def get_recent_queries():
    """Returns the 5 most recently used unique search queries."""
    try:
        pipeline = [
            {"$sort": {"timestamp": -1}},
            {"$group": {
                "_id": {"type": "$search_type", "params": "$params"},
                "timestamp": {"$first": "$timestamp"},
                "results_count": {"$first": "$results_count"},
                "total_viewed": {"$first": "$total_viewed"}
            }},
            {"$sort": {"timestamp": -1}},
            {"$limit": 5}
        ]
        return list(db_collection.aggregate(pipeline))
    except Exception as e:
        print(f"\n[MongoDB Error] Could not fetch recent queries: {e}")
        return []


def get_search_type_popularity():
    """Returns usage counts for each search type, sorted by frequency."""
    try:
        pipeline = [
            {"$group": {"_id": "$search_type", "total_uses": {"$sum": 1}}},
            {"$sort": {"total_uses": -1}}
        ]
        return list(db_collection.aggregate(pipeline))
    except Exception as e:
        print(f"\n[MongoDB Error] Could not count search type popularity: {e}")
        return []


# ---------------------------------------------------------------------------
# 5. Statistics display (tabulate)
# ---------------------------------------------------------------------------

def _format_params(q_type, params):
    """Converts a search parameter dict into a human-readable string for display.

    Args:
        q_type (str): Search type ('keyword', 'genres-years', 'actor', 'actor-years').
        params (dict): Filter parameters dict stored in MongoDB.
    Returns:
        str: Human-readable description of the applied filters.
    """
    if not params or not isinstance(params, dict):
        return "No filters"
    if q_type == "keyword":
        return f"Keyword: '{params.get('keyword', '')}'"
    elif q_type in ["genres-years", "genre"]:
        return (
            f"Genre: {params.get('genre', 'Any')} "
            f"({params.get('year_from', '...')} - {params.get('year_to', '...')})"
        )
    elif q_type in ["actor", "actor-years"]:
        return f"Actor: {params.get('actor_name', 'Unknown')}"
    return str(params)


def _show_top_queries():
    """Prints a table of the 5 most frequently repeated search queries."""
    print(
        f"\n\n{SEP}\n  {Style.BOLD}{Style.CYAN}"
        f"🎬 TOP 5 MOST FREQUENT QUERIES{Style.RESET}\n{SEP}"
    )
    top_data = []
    for idx, item in enumerate(get_top_queries(), start=1):
        g_id = item.get("_id", {})
        q_type = g_id.get("type", "-")
        top_data.append([
            idx,
            SEARCH_TYPE_LABELS.get(q_type, q_type),
            _format_params(q_type, g_id.get("params", {})),
            item.get("count", 0),
            round(item.get("avg_found", 0)),
            round(item.get("avg_viewed", 0))
        ])
    print(tabulate(
        top_data,
        headers=["#", "Search type", "Query parameters", "Count", "Avg found", "Avg viewed"],
        tablefmt="fancy_grid"
    ))


def _show_recent_queries():
    """Prints a table of the 5 most recent unique search queries."""
    print(
        f"\n\n{SEP}\n  {Style.BOLD}{Style.CYAN}"
        f"⏱ TOP 5 MOST RECENT QUERIES{Style.RESET}\n{SEP}"
    )
    recent_data = []
    for idx, item in enumerate(get_recent_queries(), start=1):
        g_id = item.get("_id", {})
        q_type = g_id.get("type", "-")

        # Format the stored datetime object for display
        dt = item.get("timestamp")
        clean_time = dt.strftime("%Y-%m-%d %H:%M:%S") if isinstance(dt, datetime) else str(dt)

        recent_data.append([
            idx,
            clean_time,
            SEARCH_TYPE_LABELS.get(q_type, q_type),
            _format_params(q_type, g_id.get("params", {})),
            item.get("results_count", 0),
            item.get("total_viewed", 0)
        ])
    print(tabulate(
        recent_data,
        headers=["#", "Date & time", "Search type", "Query parameters", "Found", "Viewed"],
        tablefmt="fancy_grid"
    ))


def _show_type_popularity():
    """Prints a table showing how many times each search type was used."""
    print(
        f"\n\n{SEP}\n  {Style.BOLD}{Style.CYAN}"
        f"📊 SEARCH TYPE POPULARITY{Style.RESET}\n{SEP}"
    )
    type_data = [
        [SEARCH_TYPE_LABELS.get(item["_id"], item["_id"]), item["total_uses"]]
        for item in get_search_type_popularity()
    ]
    print(tabulate(type_data, headers=["Search method", "Times used"], tablefmt="fancy_grid"))


def display_stats():
    """Displays all three search analytics tables."""
    _show_top_queries()
    _show_recent_queries()
    _show_type_popularity()


# ---------------------------------------------------------------------------
# 6. Main menu
# ---------------------------------------------------------------------------

def main_menu():
    """Main application menu for film search and analytics."""

    while True:
        print(f"\n{SEP}")

        # HTML title — rendered in Jupyter; falls back to plain text in a terminal
        try:
            display(HTML(
                '<h1 style="font-size: 24px; color: #008B9B; font-weight: bold; '
                'margin: 0; padding: 0; line-height: 1.3;">'
                '🎬 Film Search App — DB "sakila" 🎬'
                '</h1>'
            ))
        except Exception:
            print('🎬 Film Search App — DB "sakila" 🎬')

        print(f"\n{SEP}")

        # Live database connection status
        print(f"{Style.BOLD}Database connection status:{Style.RESET}")
        print(f"  {test_mysql_connection()}")
        print(f"  {test_mongo_connection()}")
        print(SEP)

        print("1. Search by keyword")
        print("2. Search by genre and year range")
        print("3. Search by actor and year range")
        print("4. View search statistics (MongoDB)")
        print("0. Exit")
        print(SEP)

        choice = input("Select menu option: ").strip()

        if choice == "1":
            clear_output(wait=False)
            print(f"\n{SEP}")
            print(f"  {Style.CYAN}{Style.BOLD}--- SEARCH BY KEYWORD ---{Style.RESET}")
            print(SEP)
            search_film_by_keyword()

        elif choice == "2":
            clear_output(wait=False)
            print(f"\n{SEP}")
            print(f"  {Style.CYAN}{Style.BOLD}--- SEARCH BY GENRE AND YEAR RANGE ---{Style.RESET}")
            print(SEP)
            search_film_by_genre_and_years()

        elif choice == "3":
            clear_output(wait=False)
            print(f"\n{SEP}")
            print(f"  {Style.CYAN}{Style.BOLD}--- SEARCH BY ACTOR AND YEAR RANGE ---{Style.RESET}")
            print(SEP)
            search_film_by_actor_and_year()

        elif choice == "4":
            clear_output(wait=False)
            print(f"\n{SEP}")
            print(f"  {Style.CYAN}{Style.BOLD}--- SEARCH STATISTICS FROM MONGODB ---{Style.RESET}")
            print(SEP)
            display_stats()
            print(f"\n{SEP}")
            input("Press Enter to return to the main menu...")
            clear_output(wait=False)

        elif choice == "0":
            clear_output(wait=False)
            print(f"\n{Style.GREEN}Thank you for using the system! Goodbye. 👋{Style.RESET}")
            print("Program exited.")
            break

        else:
            print(
                f"\n{Style.RED}{Style.BOLD}❌ Invalid input! "
                f"Please select an option from 0 to 4.{Style.RESET}"
            )


# ---------------------------------------------------------------------------
# Entry point
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    try:
        main_menu()
    finally:
        # Runs on both normal exit (option "0") and unexpected crashes
        try:
            mongo_client.close()
            print("MongoDB connection closed successfully.")
        except NameError:
            # mongo_client was not initialized — nothing to close
            pass


Thank you for using the system! Goodbye. 👋
Program exited.
MongoDB connection closed successfully.
